**Analyzing the Severity and Causes of Major Power Outages in the U.S.**

**Name(s)**: Samsoo Seo

**Website Link**: (your website link)

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'

# from dsc259r_utils import * # Feel free to uncomment and use this.

## Step 1: Introduction

### Introduction
This project explores the **Major Power Outages** dataset (2000-2016) in the U.S. 
My goal is to analyze how different factors—especially **Severe Weather**—impact the duration of these outages. 

**Primary Question:** "Do power outages caused by Severe Weather last longer on average than those caused by other reasons?"

## Step 2: Data Cleaning and Exploratory Data Analysis

In [25]:
df = pd.read_excel('outage.xlsx', skiprows=5)

# Define a function to combine date and time columns into a single pd.Timestamp
def to_timestamp(date_col, time_col):
    """
    Combines separate date and time columns into a single pandas Timestamp.
    """
    return pd.to_datetime(
        df[date_col].astype(str) + ' ' + df[time_col].astype(str), 
        errors='coerce'
    )

# Apply the function to start and restoration times
df['OUTAGE.START'] = to_timestamp('OUTAGE.START.DATE', 'OUTAGE.START.TIME')
df['OUTAGE.RESTORATION'] = to_timestamp('OUTAGE.RESTORATION.DATE', 'OUTAGE.RESTORATION.TIME')

# Calculate the duration of the outage in minutes
# This will be our target variable for Step 4-8
df['OUTAGE.DURATION'] = (df['OUTAGE.RESTORATION'] - df['OUTAGE.START']).dt.total_seconds() / 60



C:\Users\samso\AppData\Local\Temp\ipykernel_23364\2162984035.py:8: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.

C:\Users\samso\AppData\Local\Temp\ipykernel_23364\2162984035.py:8: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



In [26]:
# Univariate Analysis: Distribution of Outage Durations
fig_hist = px.histogram(df, x='OUTAGE.DURATION', nbins=50, 
                         title='Distribution of Power Outage Durations')
fig_hist.show()

# Bivariate Analysis: Outage Duration by Cause Category
fig_box = px.box(df, x='CAUSE.CATEGORY', y='OUTAGE.DURATION', 
                 title='Outage Duration by Cause Category (Log Scale recommended for better visibility)')
# Using log scale because duration data is often highly skewed
fig_box.update_layout(yaxis_type="log")
fig_box.show()

## Step 3: Assessment of Missingness

In [ ]:
# Changed 'barmart' to 'barmode' to fix the TypeError
fig_missing = px.bar(
    df.groupby(['DURATION_MISSING', 'CAUSE.CATEGORY']).size().reset_index(name='count'), 
    x='CAUSE.CATEGORY', 
    y='count', 
    color='DURATION_MISSING', 
    barmode='group', # Fixed the typo here
    title='Count of Outage Causes: Missing vs. Non-Missing Duration'
)

fig_missing.show()

## Step 4: Hypothesis Testing

In [37]:
import numpy as np
import pandas as pd

# Convert to lowercase and strip any leading/trailing whitespace
df['CAUSE_CLEAN'] = df['CAUSE.CATEGORY'].astype(str).str.lower().str.strip()

# Check for 'severe weather' (lowercase)
df['IS_SEVERE_WEATHER'] = df['CAUSE_CLEAN'] == 'severe weather'

# Drop rows with missing duration for this analysis
test_df = df.dropna(subset=['OUTAGE.DURATION']).copy()

group_means = test_df.groupby('IS_SEVERE_WEATHER')['OUTAGE.DURATION'].mean()

# Check if 'True' group exists now
mean_severe = group_means.get(True, 0)
mean_others = group_means.get(False, 0)
observed_diff = mean_severe - mean_others

# 5. Permutation Test
n_repetitions = 1000
shuffled_diffs = []
np.random.seed(42)

is_severe_array = test_df['IS_SEVERE_WEATHER'].values
durations_array = test_df['OUTAGE.DURATION'].values

for _ in range(n_repetitions):
    # Shuffle labels
    shuffled_labels = np.random.permutation(is_severe_array)
    
    # Calculate shuffled means
    s_mean_true = durations_array[shuffled_labels].mean() if shuffled_labels.any() else 0
    s_mean_false = durations_array[~shuffled_labels].mean() if (~shuffled_labels).any() else 0
    shuffled_diffs.append(s_mean_true - s_mean_false)

# 6. Calculate p-value
p_value_final = np.mean(np.array(shuffled_diffs) >= observed_diff)

print(f"--- Updated Hypothesis Test Results ---")
print(f"Unique values in CAUSE.CATEGORY (First 5): {df['CAUSE_CLEAN'].unique()[:5]}")
print(f"Severe Weather Mean: {mean_severe:.2f} min")
print(f"Other Causes Mean: {mean_others:.2f} min")
print(f"Observed Difference: {observed_diff:.2f} min")
print(f"P-value: {p_value_final:.4f}")

--- Updated Hypothesis Test Results ---
Unique values in CAUSE.CATEGORY (First 5): ['nan' 'severe weather' 'intentional attack'
 'system operability disruption' 'equipment failure']
Severe Weather Mean: 3883.18 min
Other Causes Mean: 1346.75 min
Observed Difference: 2536.43 min
P-value: 0.0000


## Step 5: Framing a Prediction Problem

In [38]:

# Prediction Task: Predict the duration of a major power outage (OUTAGE.DURATION)
# Problem Type: Regression
# Metric: Root Mean Squared Error (RMSE) will be used to measure prediction accuracy.

# Defining the target and features known at the time of prediction
target = 'OUTAGE.DURATION'
features = ['CAUSE.CATEGORY', 'CLIMATE.REGION', 'ANOMALY.LEVEL', 'CUSTOMERS.AFFECTED']

# Note: We must be careful not to use features that are only known AFTER the outage is resolved.
print(f"Target variable: {target}")
print(f"Features for prediction: {features}")

Target variable: OUTAGE.DURATION
Features for prediction: ['CAUSE.CATEGORY', 'CLIMATE.REGION', 'ANOMALY.LEVEL', 'CUSTOMERS.AFFECTED']


## Step 6: Baseline Model

In [7]:
# TODO

## Step 7: Final Model

In [8]:
# TODO

## Step 8: Fairness Analysis

In [9]:
# TODO